# Technical Agent Notebook 03 - Multi-Timeframe Fusion Weights

This notebook empirically determines the multi-timeframe fusion weights for the technical agent.

Rules enforced here:
- Evaluate only the held-out test period strictly after 2024-06-30.
- Use the production loading pattern from `src/agents/technical/lstm_agent.py` and `scripts/test_technical_best_models_production.py`.
- Align H4 and H1 predictions to the D1 calendar by keeping the last available prediction per day.
- Derive weights from observed test-set performance; do not hardcode them.
- Skip missing models with warnings instead of failing the notebook.

In [1]:
from __future__ import annotations

import json
import logging
import re
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from scipy.optimize import minimize
from scipy.special import softmax
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'pyproject.toml').exists()),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.shared.config import Config
from src.agents.technical import features as technical_features
from src.agents.technical.features import add_features, get_feature_names, load_pair
from src.agents.technical.lstm_agent import LSTMModel

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 120})
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
warnings.simplefilter('always')
logger = logging.getLogger('mtf_fusion_weights')
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
    logger.addHandler(handler)

TEST_CUTOFF = pd.Timestamp('2024-06-30')
TEST_START = TEST_CUTOFF + pd.Timedelta(days=1)
PAIRS = ['EURUSDm', 'GBPUSDm', 'USDCHFm', 'USDJPYm']
TIMEFRAMES = ['D1', 'H4', 'H1']
MODEL_DIR = PROJECT_ROOT / 'models' / 'best_models' / 'content' / 'best_models'
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'signals' / 'mtf_fusion_weights.json'
FEATURE_DIR = Config.DATA_DIR / 'processed' / 'ohlcv'
technical_features.DATA_DIR = FEATURE_DIR

print(f'Project root: {PROJECT_ROOT}')
print(f'OHLCV directory: {technical_features.DATA_DIR}')
print(f'Model directory: {MODEL_DIR}')
print(f'Test cutoff: {TEST_CUTOFF.date()}')

Project root: d:\SCRIPTS\FX-AlphaLab
OHLCV directory: d:\SCRIPTS\FX-AlphaLab\data\processed\ohlcv
Model directory: d:\SCRIPTS\FX-AlphaLab\models\best_models\content\best_models
Test cutoff: 2024-06-30


## Problem Framing

The target is the next D1 bar direction, `D1_next_bar_up`, and every timeframe must be judged against that same daily target.

The notebook only uses data after 2024-06-30, and the first usable prediction day for each timeframe can appear later because each sequence needs a warm-up window inside the test slice itself.

If a model or scaler file is missing, the notebook logs a warning and continues.

In [6]:
MODEL_PATTERN = re.compile(r'^lstm_(?P<pair>[A-Za-z0-9]+)_(?P<tf>D1|H4|H1)_seq(?P<seq>\d+)_best\.pth$')
DATE_PATTERN = re.compile(r'^ohlcv_(?P<pair>.+)_(?P<tf>D1|H4|H1)_(?P<start>\d{4}-\d{2}-\d{2})_(?P<end>\d{4}-\d{2}-\d{2})\.parquet$')
FEATURE_COLS = get_feature_names()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def resolve_parquet_path(pair: str, tf: str) -> Path:
    matches = list(technical_features.DATA_DIR.glob(f'ohlcv_{pair}_{tf}_*.parquet'))
    if not matches:
        raise FileNotFoundError(f'No parquet found for {pair} {tf} in {technical_features.DATA_DIR}')

    def parse_key(path: Path) -> tuple[pd.Timestamp, pd.Timestamp, str]:
        match = DATE_PATTERN.match(path.name)
        if not match:
            return (pd.Timestamp.min, pd.Timestamp.min, path.name)
        return (pd.Timestamp(match.group('start')), pd.Timestamp(match.group('end')), path.name)

    chosen = sorted(matches, key=parse_key)[-1]
    return chosen


def load_ohlcv_with_timestamp(pair: str, tf: str) -> pd.DataFrame:
    path = resolve_parquet_path(pair, tf)
    df = pd.read_parquet(path)
    if 'timestamp_utc' not in df.columns:
        raise KeyError(f'Missing timestamp_utc column in {path.name}')
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp_utc']).sort_values('timestamp_utc').copy()
    df = df.set_index('timestamp_utc')
    df.index = df.index.tz_convert(None)
    df.columns = [str(col).lower() for col in df.columns]
    expected = {'open', 'high', 'low', 'close', 'volume'}
    missing = expected.difference(df.columns)
    if missing:
        raise KeyError(f'Missing columns in {path.name}: {sorted(missing)}')
    return df

technical_features.find_parquet = resolve_parquet_path
technical_features.load_pair = load_ohlcv_with_timestamp
load_pair = load_ohlcv_with_timestamp

@dataclass(frozen=True)
class ModelSpec:
    pair: str
    timeframe: str
    seq_len: int
    model_path: Path
    scaler_path: Path

def parse_model_spec(model_path: Path) -> ModelSpec:
    match = MODEL_PATTERN.match(model_path.name)
    if not match:
        raise ValueError(f'Unexpected model filename: {model_path.name}')
    pair = match.group('pair')
    timeframe = match.group('tf')
    seq_len = int(match.group('seq'))
    scaler_path = model_path.with_name(model_path.name.replace('_best.pth', '_scaler.pkl'))
    return ModelSpec(pair=pair, timeframe=timeframe, seq_len=seq_len, model_path=model_path, scaler_path=scaler_path)

def load_model_artifact(spec: ModelSpec) -> tuple[LSTMModel, object]:
    checkpoint = torch.load(spec.model_path, map_location=DEVICE, weights_only=False)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        config = checkpoint.get('model_config', {})
        state_dict = checkpoint['model_state_dict']
        input_size = int(config.get('input_size', len(FEATURE_COLS)))
        hidden_size = int(config.get('hidden_size', 128))
        num_layers = int(config.get('num_layers', 2))
        dropout = float(config.get('dropout', 0.3))
    else:
        state_dict = checkpoint
        input_size = len(FEATURE_COLS)
        hidden_size = 128
        num_layers = 2
        dropout = 0.3

    model = LSTMModel(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    scaler = joblib.load(spec.scaler_path)
    return model, scaler

def prepare_test_frame(pair: str, timeframe: str) -> pd.DataFrame:
    raw = load_pair(pair, timeframe)
    test_raw = raw.loc[raw.index > TEST_CUTOFF].copy()
    if test_raw.empty:
        raise ValueError(f'No rows after test cutoff for {pair} {timeframe}')
    return add_features(test_raw)

def sequence_probabilities(df: pd.DataFrame, spec: ModelSpec, model: LSTMModel, scaler: object) -> pd.DataFrame:
    if len(df) <= spec.seq_len:
        raise ValueError(f'Not enough rows for {spec.pair} {spec.timeframe} with seq_len={spec.seq_len}')

    scaled = df.copy()
    scaled[FEATURE_COLS] = scaler.transform(scaled[FEATURE_COLS])

    sequences = np.stack([scaled[FEATURE_COLS].to_numpy()[i - spec.seq_len:i] for i in range(spec.seq_len, len(scaled))])
    timestamps = scaled.index[spec.seq_len:]
    targets = scaled['target'].iloc[spec.seq_len:].to_numpy(dtype=int)

    x_tensor = torch.tensor(sequences, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = model(x_tensor)
        probs = torch.sigmoid(logits).detach().cpu().numpy().astype(float)

    out = pd.DataFrame({
        'timestamp_utc': pd.to_datetime(timestamps),
        'date': pd.to_datetime(timestamps).normalize(),
        'prob_up': probs,
        'target': targets,
    })
    return out

def daily_last_prediction(pred_df: pd.DataFrame, rename_to: str) -> pd.DataFrame:
    daily = pred_df.sort_values('timestamp_utc').groupby('date', as_index=False).tail(1).copy()
    return daily[['date', 'prob_up']].rename(columns={'prob_up': rename_to})

def build_pair_table(pair: str) -> pd.DataFrame | None:
    pair_tables: dict[str, pd.DataFrame] = {}
    target_table: pd.DataFrame | None = None

    for timeframe in TIMEFRAMES:
        candidates = sorted(MODEL_DIR.glob(f'lstm_{pair}_{timeframe}_seq*_best.pth'))
        if not candidates:
            warnings.warn(f'Missing model for {pair} {timeframe}; skipping pair.')
            return None

        spec = parse_model_spec(candidates[0])
        if not spec.scaler_path.exists():
            warnings.warn(f'Missing scaler for {spec.model_path.name}; skipping pair.')
            return None

        try:
            frame = prepare_test_frame(pair, timeframe)
            model, scaler = load_model_artifact(spec)
            pred_df = sequence_probabilities(frame, spec, model, scaler)
            daily = daily_last_prediction(pred_df, f'{timeframe}_prob_up')
            pair_tables[timeframe] = daily
            if timeframe == 'D1':
                target_table = pred_df.sort_values('timestamp_utc').groupby('date', as_index=False).tail(1)[['date', 'target']].copy()
        except Exception as exc:  # noqa: BLE001
            warnings.warn(f'Failed to load or score {pair} {timeframe}: {exc}')
            return None

    merged = target_table
    for timeframe in TIMEFRAMES:
        merged = merged.merge(pair_tables[timeframe], on='date', how='inner')

    merged = merged.sort_values('date').reset_index(drop=True)
    merged.insert(0, 'pair', pair)
    merged.rename(columns={'target': 'D1_next_bar_up'}, inplace=True)
    return merged

print('Helpers ready.')

Helpers ready.


## Data Availability Check

The OHLCV files do extend well past the requested cutoff, so a held-out test period does exist. The important detail is that the parquet stores `timestamp_utc` as a column, not as the index, so the notebook must build the datetime index explicitly before any temporal split or daily alignment is trustworthy.

In [7]:
coverage_rows: list[dict[str, object]] = []

for pair in PAIRS:
    for timeframe in TIMEFRAMES:
        try:
            df = load_pair(pair, timeframe)
            if df.index.tz is not None:
                df.index = df.index.tz_convert(None)
            coverage_rows.append(
                {
                    'pair': pair,
                    'timeframe': timeframe,
                    'start': df.index.min(),
                    'end': df.index.max(),
                    'rows': len(df),
                }
            )
        except Exception as exc:  # noqa: BLE001
            coverage_rows.append(
                {
                    'pair': pair,
                    'timeframe': timeframe,
                    'start': pd.NaT,
                    'end': pd.NaT,
                    'rows': 0,
                    'error': str(exc),
                }
            )

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

print('Rows strictly after 2024-06-30 by pair/timeframe:')
for pair in PAIRS:
    for timeframe in TIMEFRAMES:
        df = load_pair(pair, timeframe)
        if df.index.tz is not None:
            df.index = df.index.tz_convert(None)
        after_cutoff = int((df.index > TEST_CUTOFF).sum())
        print(f'{pair} {timeframe}: {after_cutoff}')

,pair,timeframe,start,end,rows
0,EURUSDm,D1,2021-01-03 00:00:00,2025-12-30 00:00:00,1560
1,EURUSDm,H4,2021-01-03 20:00:00,2025-12-30 20:00:00,8035
2,EURUSDm,H1,2021-01-03 22:00:00,2025-12-30 23:00:00,31086
3,GBPUSDm,D1,2021-01-03 00:00:00,2025-12-30 00:00:00,1560
4,GBPUSDm,H4,2021-01-03 20:00:00,2025-12-30 20:00:00,8035
5,GBPUSDm,H1,2021-01-03 22:00:00,2025-12-30 23:00:00,31086
6,USDCHFm,D1,2021-01-03 00:00:00,2025-12-30 00:00:00,1560
7,USDCHFm,H4,2021-01-03 20:00:00,2025-12-30 20:00:00,8035
8,USDCHFm,H1,2021-01-03 22:00:00,2025-12-30 23:00:00,31085
9,USDJPYm,D1,2021-01-03 00:00:00,2025-12-30 00:00:00,1560


Rows strictly after 2024-06-30 by pair/timeframe:
EURUSDm D1: 469
EURUSDm H4: 2407
EURUSDm H1: 9299
GBPUSDm D1: 469
GBPUSDm H4: 2407
GBPUSDm H1: 9299
USDCHFm D1: 469
USDCHFm H4: 2407
USDCHFm H1: 9299
USDJPYm D1: 469
USDJPYm H4: 2407
USDJPYm H1: 9299


## Section 1 - Generate Daily-Aligned Predictions

After fixing the loader, the test slice is real: EURUSDm, GBPUSDm, and USDCHFm produce 389 aligned daily rows, while USDJPYm produces 419. The later start dates are caused by the sequence warm-up inside the test window and by weekend gaps in the daily alignment. The first row in each pair table is the last available prediction per day from D1, H4, and H1 merged onto the same calendar date.

In [8]:
pair_tables: dict[str, pd.DataFrame] = {}
skipped_pairs: list[str] = []

model_files = sorted(MODEL_DIR.glob('lstm_*_best.pth'))
print(f'Found {len(model_files)} model artifacts in {MODEL_DIR}')
print(f'Using only data after {TEST_START.date()} for evaluation')

for pair in PAIRS:
    table = build_pair_table(pair)
    if table is None or table.empty:
        skipped_pairs.append(pair)
        continue
    pair_tables[pair] = table

if skipped_pairs:
    print('Skipped pairs:', skipped_pairs)

for pair, table in pair_tables.items():
    display(table.head(5))
    print(f'{pair}: {len(table)} aligned daily rows')

Found 12 model artifacts in d:\SCRIPTS\FX-AlphaLab\models\best_models\content\best_models
Using only data after 2024-07-01 for evaluation


d:\SCRIPTS\FX-AlphaLab\.venv\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\SCRIPTS\FX-AlphaLab\.venv\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\SCRIPTS\FX-AlphaLab\.venv\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to br

,pair,date,D1_next_bar_up,D1_prob_up,H4_prob_up,H1_prob_up
0,EURUSDm,2024-10-02,0,0.481465,0.511857,0.500274
1,EURUSDm,2024-10-03,0,0.481954,0.511964,0.497725
2,EURUSDm,2024-10-04,0,0.482454,0.512040,0.504652
3,EURUSDm,2024-10-06,1,0.483052,0.512079,0.503663
4,EURUSDm,2024-10-07,1,0.483654,0.511870,0.494714


EURUSDm: 389 aligned daily rows


,pair,date,D1_next_bar_up,D1_prob_up,H4_prob_up,H1_prob_up
0,GBPUSDm,2024-10-02,0,0.481071,0.505704,0.496327
1,GBPUSDm,2024-10-03,0,0.481105,0.507914,0.498249
2,GBPUSDm,2024-10-04,0,0.481150,0.504246,0.495191
3,GBPUSDm,2024-10-06,0,0.481138,0.503732,0.495306
4,GBPUSDm,2024-10-07,1,0.481079,0.503222,0.495442


GBPUSDm: 389 aligned daily rows


,pair,date,D1_next_bar_up,D1_prob_up,H4_prob_up,H1_prob_up
0,USDCHFm,2024-10-02,1,0.507970,0.498232,0.490610
1,USDCHFm,2024-10-03,1,0.508174,0.498499,0.491071
2,USDCHFm,2024-10-04,1,0.508554,0.498762,0.489194
3,USDCHFm,2024-10-06,0,0.509041,0.498788,0.490335
4,USDCHFm,2024-10-07,1,0.509488,0.499407,0.494249


USDCHFm: 389 aligned daily rows


,pair,date,D1_next_bar_up,D1_prob_up,H4_prob_up,H1_prob_up
0,USDJPYm,2024-08-28,1,0.561881,0.529462,0.519468
1,USDJPYm,2024-08-29,1,0.561988,0.530146,0.519343
2,USDJPYm,2024-08-30,1,0.562368,0.531286,0.518892
3,USDJPYm,2024-09-01,1,0.563279,0.531540,0.519002
4,USDJPYm,2024-09-02,0,0.564388,0.531820,0.519083


USDJPYm: 419 aligned daily rows


## Section 2 - Per-Timeframe Accuracy on the Daily Target

The test-set evidence is weak and tightly clustered around chance. D1 has the highest mean accuracy across pairs, but H4 is competitive on F1 for some pairs and H1 occasionally has the best ROC-AUC. Spearman correlations are close to zero, and Brier scores sit near 0.25, which means these models are not producing strongly separated probability estimates on the daily target.

In [9]:
metric_rows: list[dict[str, object]] = []

for pair, table in pair_tables.items():
    for timeframe in TIMEFRAMES:
        prob_col = f'{timeframe}_prob_up'
        y_true = table['D1_next_bar_up'].to_numpy(dtype=int)
        y_prob = table[prob_col].to_numpy(dtype=float)
        y_pred = (y_prob >= 0.5).astype(int)

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = np.nan

        rho = spearmanr(y_prob, y_true).correlation
        metric_rows.append({
            'pair': pair,
            'timeframe': timeframe,
            'accuracy': accuracy_score(y_true, y_pred),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': auc,
            'brier': brier_score_loss(y_true, y_prob),
            'spearman_rho': rho,
            'n_days': len(table),
        })

summary_df = pd.DataFrame(metric_rows).sort_values(['pair', 'timeframe']).reset_index(drop=True)
display(summary_df)

print('Mean metrics by timeframe across pairs:')
display(summary_df.groupby('timeframe')[['accuracy', 'f1', 'roc_auc', 'brier', 'spearman_rho']].mean().round(4))

,pair,timeframe,accuracy,f1,roc_auc,brier,spearman_rho,n_days
0,EURUSDm,D1,0.514139,0.000000,0.506561,0.250010,0.011359,389
1,EURUSDm,H1,0.493573,0.404834,0.479471,0.250146,-0.035543,389
2,EURUSDm,H4,0.485861,0.653979,0.500529,0.250424,0.000916,389
3,GBPUSDm,D1,0.503856,0.000000,0.505895,0.250221,0.010210,389
4,GBPUSDm,H1,0.498715,0.020101,0.509332,0.249955,0.016163,389
5,GBPUSDm,H4,0.514139,0.473538,0.544438,0.249717,0.076967,389
6,USDCHFm,D1,0.516710,0.681356,0.509659,0.249689,0.016721,389
7,USDCHFm,H1,0.483290,0.000000,0.520192,0.250290,0.034953,389
8,USDCHFm,H4,0.493573,0.308772,0.500370,0.250027,0.000641,389
9,USDJPYm,D1,0.529833,0.692668,0.512485,0.251078,0.021585,419


Mean metrics by timeframe across pairs:


,accuracy,f1,roc_auc,brier,spearman_rho
timeframe,,,,,
D1,0.5161,0.3435,0.5086,0.2502,0.0150
H1,0.5014,0.2794,0.5100,0.2499,0.0172
H4,0.5059,0.5322,0.5084,0.2498,0.0145


## Section 3 - Three Fusion Weight Methods

The three learned methods do not uncover a materially different fusion structure. The accuracy-proportional method nudges weights toward the slightly stronger timeframe in each pair, but the constrained optimizer and the logistic-regression meta-model collapse almost exactly to equal weights. That is the expected outcome when the inputs are only weakly predictive and highly redundant.

In [10]:
def normalize_weights(weights: dict[str, float]) -> dict[str, float]:
    values = np.array([weights['D1'], weights['H4'], weights['H1']], dtype=float)
    total = values.sum()
    if total <= 0 or not np.isfinite(total):
        values = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
    else:
        values = values / total
    return {'D1': float(values[0]), 'H4': float(values[1]), 'H1': float(values[2])}

def fused_metrics(table: pd.DataFrame, weights: dict[str, float]) -> dict[str, float]:
    probs = (
        weights['D1'] * table['D1_prob_up'].to_numpy(dtype=float)
        + weights['H4'] * table['H4_prob_up'].to_numpy(dtype=float)
        + weights['H1'] * table['H1_prob_up'].to_numpy(dtype=float)
    )
    y_true = table['D1_next_bar_up'].to_numpy(dtype=int)
    y_pred = (probs >= 0.5).astype(int)
    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = np.nan
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': auc,
        'brier': brier_score_loss(y_true, probs),
    }

def method_a_weights(pair: str) -> dict[str, float]:
    row = summary_df[summary_df['pair'] == pair].set_index('timeframe')['accuracy'].to_dict()
    return normalize_weights({'D1': row['D1'], 'H4': row['H4'], 'H1': row['H1']})

def method_b_weights(table: pd.DataFrame) -> tuple[dict[str, float], dict[str, object]]:
    y_true = table['D1_next_bar_up'].to_numpy(dtype=int)
    X = table[['D1_prob_up', 'H4_prob_up', 'H1_prob_up']].to_numpy(dtype=float)

    def objective(w: np.ndarray) -> float:
        w = np.clip(w, 0.0, 1.0)
        total = w.sum()
        if total <= 0:
            w = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
        else:
            w = w / total
        probs = X @ w
        preds = (probs >= 0.5).astype(int)
        return -accuracy_score(y_true, preds)

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
    bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
    start = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
    result = minimize(objective, start, method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter': 200, 'ftol': 1e-9})
    weights = start if not result.success else np.clip(result.x, 0.0, 1.0)
    weights = weights / weights.sum()
    return ({'D1': float(weights[0]), 'H4': float(weights[1]), 'H1': float(weights[2])}, {'success': result.success, 'message': result.message, 'fun': float(result.fun)})

def method_c_weights(table: pd.DataFrame) -> tuple[dict[str, float], dict[str, object]]:
    y = table['D1_next_bar_up'].to_numpy(dtype=int)
    X = table[['D1_prob_up', 'H4_prob_up', 'H1_prob_up']].to_numpy(dtype=float)
    cv = TimeSeriesSplit(n_splits=3)
    model = LogisticRegressionCV(
        Cs=np.logspace(-3, 3, 7),
        cv=cv,
        fit_intercept=False,
        solver='lbfgs',
        max_iter=1000,
        scoring='accuracy',
    )
    model.fit(X, y)
    coefs = model.coef_.ravel().astype(float)
    weights = softmax(coefs)
    return ({'D1': float(weights[0]), 'H4': float(weights[1]), 'H1': float(weights[2])}, {'C': float(model.C_[0]), 'coef': coefs.tolist()})

method_rows: list[dict[str, object]] = []
weight_records: list[dict[str, object]] = []
comparison_rows: list[dict[str, object]] = []

for pair, table in pair_tables.items():
    weights_a = method_a_weights(pair)
    weights_b, meta_b = method_b_weights(table)
    weights_c, meta_c = method_c_weights(table)
    equal_weights = {'D1': 1 / 3, 'H4': 1 / 3, 'H1': 1 / 3}

    methods = {
        'A_accuracy_proportional': weights_a,
        'B_slsqp_accuracy_max': weights_b,
        'C_logistic_regression_cv': weights_c,
        'equal_weights': equal_weights,
        'D1_only': {'D1': 1.0, 'H4': 0.0, 'H1': 0.0},
    }

    for method_name, weights in methods.items():
        metrics = fused_metrics(table, weights)
        comparison_rows.append({
            'pair': pair,
            'method': method_name,
            'accuracy': metrics['accuracy'],
            'f1': metrics['f1'],
            'roc_auc': metrics['roc_auc'],
            'brier': metrics['brier'],
        })

    weight_records.append({'pair': pair, 'method': 'A_accuracy_proportional', **weights_a})
    weight_records.append({'pair': pair, 'method': 'B_slsqp_accuracy_max', **weights_b})
    weight_records.append({'pair': pair, 'method': 'C_logistic_regression_cv', **weights_c})
    weight_records.append({'pair': pair, 'method': 'equal_weights', **equal_weights})

    method_rows.append({'pair': pair, 'method': 'B_slsqp_accuracy_max', **meta_b})
    method_rows.append({'pair': pair, 'method': 'C_logistic_regression_cv', **meta_c})

weights_df = pd.DataFrame(weight_records)
comparison_df = pd.DataFrame(comparison_rows)
opt_meta_df = pd.DataFrame(method_rows)

display(weights_df)
display(comparison_df.sort_values(['pair', 'method']))
display(opt_meta_df)

,pair,method,D1,H4,H1
0,EURUSDm,A_accuracy_proportional,0.344234,0.325301,0.330465
1,EURUSDm,B_slsqp_accuracy_max,0.333333,0.333333,0.333333
2,EURUSDm,C_logistic_regression_cv,0.333358,0.333315,0.333327
3,EURUSDm,equal_weights,0.333333,0.333333,0.333333
4,GBPUSDm,A_accuracy_proportional,0.332203,0.338983,0.328814
5,GBPUSDm,B_slsqp_accuracy_max,0.333333,0.333333,0.333333
6,GBPUSDm,C_logistic_regression_cv,0.333331,0.333341,0.333328
7,GBPUSDm,equal_weights,0.333333,0.333333,0.333333
8,USDCHFm,A_accuracy_proportional,0.345955,0.330465,0.323580
9,USDCHFm,B_slsqp_accuracy_max,0.333333,0.333333,0.333333


,pair,method,accuracy,f1,roc_auc,brier
0,EURUSDm,A_accuracy_proportional,0.503856,0.303249,0.485529,0.250018
1,EURUSDm,B_slsqp_accuracy_max,0.501285,0.316901,0.485661,0.250023
2,EURUSDm,C_logistic_regression_cv,0.501285,0.316901,0.485661,0.250023
4,EURUSDm,D1_only,0.514139,0.000000,0.506561,0.250010
3,EURUSDm,equal_weights,0.501285,0.316901,0.485661,0.250023
5,GBPUSDm,A_accuracy_proportional,0.503856,0.000000,0.533256,0.249899
6,GBPUSDm,B_slsqp_accuracy_max,0.503856,0.000000,0.532806,0.249901
7,GBPUSDm,C_logistic_regression_cv,0.503856,0.000000,0.532806,0.249901
9,GBPUSDm,D1_only,0.503856,0.000000,0.505895,0.250221
8,GBPUSDm,equal_weights,0.503856,0.000000,0.532806,0.249901


,pair,method,success,message,fun,C,coef
0,EURUSDm,B_slsqp_accuracy_max,True,Optimization terminated successfully,-0.501285,NaN,NaN
1,EURUSDm,C_logistic_regression_cv,NaN,NaN,NaN,0.001,"[-0.00248950326316756, -0.0026189758637090803,..."
2,GBPUSDm,B_slsqp_accuracy_max,True,Optimization terminated successfully,-0.503856,NaN,NaN
3,GBPUSDm,C_logistic_regression_cv,NaN,NaN,NaN,0.001,"[-0.0006743280878681507, -0.000645283960822156..."
4,USDCHFm,B_slsqp_accuracy_max,True,Optimization terminated successfully,-0.521851,NaN,NaN
5,USDCHFm,C_logistic_regression_cv,NaN,NaN,NaN,0.001,"[0.0030999383978489387, 0.003025224548192499, ..."
6,USDJPYm,B_slsqp_accuracy_max,True,Optimization terminated successfully,-0.529833,NaN,NaN
7,USDJPYm,C_logistic_regression_cv,NaN,NaN,NaN,0.001,"[0.006603595588147319, 0.006053693359886359, 0..."


## Section 4 - Compare and Recommend

Equal weights stay the recommendation because the best learned method only improves mean accuracy by 0.0026 over equal weights, which is not enough to justify a more complex rule on this test set. The pair-level weights are also completely stable in practice: every pair ends up at 1/3, 1/3, 1/3 after the recommendation rule is applied. That means a single global weight vector is defensible.

In [11]:
mean_compare = comparison_df.groupby('method')[['accuracy', 'f1', 'roc_auc', 'brier']].mean().sort_values('accuracy', ascending=False)
display(mean_compare)

pair_method_matrix = comparison_df.pivot(index='pair', columns='method', values='accuracy')
display(pair_method_matrix)

best_candidate = mean_compare.index[0]
equal_accuracy = float(mean_compare.loc['equal_weights', 'accuracy'])
best_accuracy = float(mean_compare.loc[best_candidate, 'accuracy'])
accuracy_gain = best_accuracy - equal_accuracy
recommended_method = best_candidate if accuracy_gain >= 0.01 else 'equal_weights'

selected_weights_by_pair: dict[str, dict[str, float]] = {}
for pair in pair_tables:
    if recommended_method == 'equal_weights':
        selected_weights_by_pair[pair] = {'D1': 1 / 3, 'H4': 1 / 3, 'H1': 1 / 3}
    elif recommended_method == 'A_accuracy_proportional':
        selected_weights_by_pair[pair] = method_a_weights(pair)
    elif recommended_method == 'B_slsqp_accuracy_max':
        selected_weights_by_pair[pair] = weights_df[(weights_df['pair'] == pair) & (weights_df['method'] == 'B_slsqp_accuracy_max')][['D1', 'H4', 'H1']].iloc[0].to_dict()
    else:
        selected_weights_by_pair[pair] = weights_df[(weights_df['pair'] == pair) & (weights_df['method'] == 'C_logistic_regression_cv')][['D1', 'H4', 'H1']].iloc[0].to_dict()

selected_weights_df = pd.DataFrame([{'pair': pair, **weights} for pair, weights in selected_weights_by_pair.items()])
display(selected_weights_df)

stability_rows = []
for timeframe in TIMEFRAMES:
    values = selected_weights_df[timeframe].to_numpy(dtype=float)
    stability_rows.append({
        'timeframe': timeframe,
        'mean_weight': float(values.mean()),
        'std_weight': float(values.std(ddof=0)),
        'min_weight': float(values.min()),
        'max_weight': float(values.max()),
        'range_weight': float(values.max() - values.min()),
    })
stability_df = pd.DataFrame(stability_rows)
display(stability_df)

global_weights = normalize_weights({
    'D1': float(selected_weights_df['D1'].mean()),
    'H4': float(selected_weights_df['H4'].mean()),
    'H1': float(selected_weights_df['H1'].mean()),
})

print(f'Recommended method: {recommended_method}')
print(f'Equal-weight mean accuracy: {equal_accuracy:.4f}')
print(f'Best candidate mean accuracy: {best_accuracy:.4f}')
print(f'Accuracy gain over equal weights: {accuracy_gain:.4f}')
print(f'Global weights: {global_weights}')

,accuracy,f1,roc_auc,brier
method,,,,
A_accuracy_proportional,0.516777,0.370106,0.508558,0.249768
D1_only,0.516134,0.343506,0.508650,0.250250
B_slsqp_accuracy_max,0.514206,0.370290,0.508611,0.249772
C_logistic_regression_cv,0.514206,0.370290,0.508605,0.249772
equal_weights,0.514206,0.370290,0.508611,0.249772


method,A_accuracy_proportional,B_slsqp_accuracy_max,C_logistic_regression_cv,D1_only,equal_weights
pair,,,,,
EURUSDm,0.503856,0.501285,0.501285,0.514139,0.501285
GBPUSDm,0.503856,0.503856,0.503856,0.503856,0.503856
USDCHFm,0.529563,0.521851,0.521851,0.516710,0.521851
USDJPYm,0.529833,0.529833,0.529833,0.529833,0.529833


,pair,D1,H4,H1
0,EURUSDm,0.333333,0.333333,0.333333
1,GBPUSDm,0.333333,0.333333,0.333333
2,USDCHFm,0.333333,0.333333,0.333333
3,USDJPYm,0.333333,0.333333,0.333333


,timeframe,mean_weight,std_weight,min_weight,max_weight,range_weight
0,D1,0.333333,0.0,0.333333,0.333333,0.0
1,H4,0.333333,0.0,0.333333,0.333333,0.0
2,H1,0.333333,0.0,0.333333,0.333333,0.0


Recommended method: equal_weights
Equal-weight mean accuracy: 0.5142
Best candidate mean accuracy: 0.5168
Accuracy gain over equal weights: 0.0026
Global weights: {'D1': 0.3333333333333333, 'H4': 0.3333333333333333, 'H1': 0.3333333333333333}


## Section 5 - Save the Recommended Weights

The JSON artifact has been written to `outputs/signals/mtf_fusion_weights.json`. It records equal weights as the final recommendation, alongside the per-pair vectors and the held-out period used for selection.

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

payload = {
    'method': recommended_method,
    'global_weights': {k: round(v, 6) for k, v in global_weights.items()},
    'per_pair_weights': {
        pair: {k: round(v, 6) for k, v in weights.items()}
        for pair, weights in selected_weights_by_pair.items()
    },
    'test_period': '2024-07-01 to end',
    'note': (
        'Weights were derived from the held-out test slice only. '
        f'Equal weights won the recommendation rule because the best candidate improved accuracy by {accuracy_gain:.4f}.'
        if recommended_method == 'equal_weights'
        else f'Recommended method {recommended_method} exceeded equal weights by {accuracy_gain:.4f} on mean pair accuracy.'
    ),
}

with OUTPUT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2)

print(f'Saved weights to {OUTPUT_PATH}')
print(json.dumps(payload, indent=2))

Saved weights to d:\SCRIPTS\FX-AlphaLab\outputs\signals\mtf_fusion_weights.json
{
  "method": "equal_weights",
  "global_weights": {
    "D1": 0.333333,
    "H4": 0.333333,
    "H1": 0.333333
  },
  "per_pair_weights": {
    "EURUSDm": {
      "D1": 0.333333,
      "H4": 0.333333,
      "H1": 0.333333
    },
    "GBPUSDm": {
      "D1": 0.333333,
      "H4": 0.333333,
      "H1": 0.333333
    },
    "USDCHFm": {
      "D1": 0.333333,
      "H4": 0.333333,
      "H1": 0.333333
    },
    "USDJPYm": {
      "D1": 0.333333,
      "H4": 0.333333,
      "H1": 0.333333
    }
  },
  "test_period": "2024-07-01 to end",
  "note": "Weights were derived from the held-out test slice only. Equal weights won the recommendation rule because the best candidate improved accuracy by 0.0026."
}


## Conclusions

The empirical result is simple: the three timeframes are individually weak on the daily target, and none of the learned fusion methods beats equal weights by a meaningful margin. Because the learned coefficients collapse to the simplex center and the optimizer lands there as well, equal weights are the cleanest coordinator rule for now.

The useful next step is not more weight tuning; it is to look for better pair-specific features or a richer meta-labeling target that can separate the timeframes more clearly.